# Phase 2 — Paires d'entraînement synthétiques

On dégrade du gcf propre vers du pseudo-Whisper-ht en appliquant les règles à l'envers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.config import CFG


## 1-3. Génération

In [ ]:
from src.phase2_synthetic import build_parser, generate
stats = generate(build_parser().parse_args(['--n', '80000', '--variants-per-sentence', '2']))
stats


## Contrôle qualité : à quoi ressemblent les paires ?

À vérifier à l'œil : le pseudo-ht doit être *plausible*. S'il est trop propre, le correcteur n'apprendra rien ; s'il est illisible, il apprendra à halluciner.

In [ ]:
import pandas as pd
df = pd.read_csv(CFG.train_pairs)
print(len(df))
df.sample(15)


## 4. Vérification de l'absence de fuite entre splits

In [ ]:
from src.phase2_synthetic import _fingerprint
tr = set(_fingerprint(t) for t in pd.read_csv(CFG.train_pairs).target_gcf)
te = set(_fingerprint(t) for t in pd.read_csv(CFG.test_pairs).target_gcf)
print('cibles partagées train/test :', len(tr & te), '(doit être 0)')


## Distribution du WER induit par la corruption

In [ ]:
from src.metrics import wer
import numpy as np
s = df.sample(2000)
w = [wer(t, x) for t, x in zip(s.target_gcf, s.source_pseudo_ht)]
print('WER moyen de la corruption :', round(float(np.mean(w)), 4))
print('quantiles :', np.round(np.quantile(w, [.1,.25,.5,.75,.9]), 3))
